In [2]:
from datasets import load_dataset  #Import Huggingface's load_dataset function
import json

In [6]:
!pip install "datasets==2.20.0"

INFO: pip is looking at multiple versions of multiprocess to determine which version is compatible with other requirements. This could take a while.
  Using cached multiprocess-0.70.18-py313-none-any.whl.metadata (7.2 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 547.8/547.8 kB 13.1 MB/s  0:00:00
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2026.2.0
    Uninstalling fsspec-2026.2.0:
      Successfully uninstalled fsspec-2026.2.0
  Attempting uninstall: dill
    Found existing installation: dill 0.4.1
    Uninstalling dill-0.4.1:
      Successfully uninstalled dill-0.4.1
  Attempting uninstall: multiprocess━╺━━━━━━━━━━━━━━━━━━━━━━━ 2/5 [dill]
    Found existing installation: multiprocess 0.70.19━━━━━━━━━━━━━━━━━━━━━━━ 2/5 [dill]
    Uninstalling multiprocess-0.70.19:╺━━━━━━━━━━━━━━━━━━━━━━━ 2/5 [dill]
      Successfully uninstalled multiprocess-0.70.190m━━━━━━━━━━━━━━━━━━━━━━━ 2/5 [dill]
  Attempting uninstall: datasets━╺━━━━━━━━━━━━━━━━━━━━━━━ 2/5 [dill]
   

In [3]:
corpus=load_dataset("allenai/scifact", "corpus", trust_remote_code=True)["train"]

print(type(corpus))
print(f"Number of documents: {len(corpus)}")

Generating train split:   0%|          | 0/5183 [00:00<?, ? examples/s]

<class 'datasets.arrow_dataset.Dataset'>
Number of documents: 5183


In [4]:
doc=corpus[0]
print("Keys in a document:", doc.keys())
print()
print("doc_id:", doc["doc_id"])
print("title:", doc["title"])
print("structured:", doc["structured"])
print()
print("abstract (first 3 sentences):")
for i, sentence in enumerate(doc["abstract"][:3]):
    print(f"  [{i}] {sentence}")

Keys in a document: dict_keys(['doc_id', 'title', 'abstract', 'structured'])

doc_id: 4983
title: Microstructural development of human newborn cerebral white matter assessed in vivo by diffusion tensor magnetic resonance imaging.
structured: False

abstract (first 3 sentences):
  [0] Alterations of the architecture of cerebral white matter in the developing human brain can affect cortical development and result in functional disabilities.
  [1] A line scan diffusion-weighted magnetic resonance imaging (MRI) sequence with diffusion tensor analysis was applied to measure the apparent diffusion coefficient, to calculate relative anisotropy, and to delineate three-dimensional fiber architecture in cerebral white matter in preterm (n = 17) and full-term infants (n = 7).
  [2] To assess effects of prematurity on cerebral white matter development, early gestation preterm infants (n = 10) were studied a second time at term.


In [5]:
claims_train = load_dataset("allenai/scifact", "claims")["train"]
claims_val = load_dataset("allenai/scifact", "claims")["validation"]

print(f"Train claims: {len(claims_train)}")
print(f"Validation claims: {len(claims_val)}")

Generating train split:   0%|          | 0/1261 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/300 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/450 [00:00<?, ? examples/s]

Train claims: 1261
Validation claims: 450


In [6]:
claim = claims_val[0]
print("Keys in a claim:", claim.keys())
print()
print("id:", claim["id"])
print("claim:", claim["claim"])
print("evidence_doc_id:", claim["evidence_doc_id"])
print("evidence_label:", claim["evidence_label"])
print("evidence_sentences:", claim["evidence_sentences"])
print("cited_doc_ids:", claim["cited_doc_ids"])

Keys in a claim: dict_keys(['id', 'claim', 'evidence_doc_id', 'evidence_label', 'evidence_sentences', 'cited_doc_ids'])

id: 1
claim: 0-dimensional biomaterials show inductive properties.
evidence_doc_id: 
evidence_label: 
evidence_sentences: []
cited_doc_ids: [31715818]


In [7]:
# Build a set of all doc_ids in the corpus for fast lookup
corpus_ids = set(doc["doc_id"] for doc in corpus)
print(f"Unique doc IDs in corpus: {len(corpus_ids)}")

# Check validation claims
missing = 0
no_evidence = 0

for claim in claims_val:
    if not claim["evidence_doc_id"]:
        no_evidence += 1
        continue
    
    doc_id = int(claim["evidence_doc_id"])
    if doc_id not in corpus_ids:
        missing += 1
        print(f"  Missing: claim {claim['id']} references doc {doc_id}")

print(f"\nClaims with no evidence_doc_id: {no_evidence}")
print(f"Claims referencing a missing doc: {missing}")

Unique doc IDs in corpus: 5183

Claims with no evidence_doc_id: 112
Claims referencing a missing doc: 0


In [9]:
import numpy as np

lengths=[]
for doc in corpus:
    full_text=" ".join(doc["abstract"])
    word_count=len(full_text.split())
    lengths.append(word_count)
lengths=np.array(lengths)

print(f"Average abstract length: {lengths.mean()} words")
print(f"Median: {np.median(lengths)} words")
print(f"Max {lengths.max()} words")
print(f"Min {lengths.min()} word")
print(f"Std dev: {lengths.std():.0f} words")
print()
print(f"Abstracts under 512 words: {(lengths < 512).sum()} ({(lengths < 512).mean():.1%})")
print(f"Abstracts under 256 words: {(lengths < 256).sum()} ({(lengths < 256).mean():.1%})")


Average abstract length: 201.81092031641907 words
Median: 192.0 words
Max 1524 words
Min 26 word
Std dev: 86 words

Abstracts under 512 words: 5150 (99.4%)
Abstracts under 256 words: 4158 (80.2%)


In [10]:
# Convert to plain Python lists so they are easy to work with
# across all future notebooks

corpus_clean = []
for doc in corpus:
    corpus_clean.append({
        "doc_id": doc["doc_id"],
        "title": doc["title"],
        "abstract_sentences": doc["abstract"],           # list of sentences
        "abstract_text": " ".join(doc["abstract"]),      # joined string
        "structured": doc["structured"]
    })

claims_clean = []
for claim in claims_val:
    if claim["evidence_doc_id"]:  # only keep claims with ground truth
        claims_clean.append({
            "id": claim["id"],
            "claim": claim["claim"],
            "evidence_doc_id": int(claim["evidence_doc_id"]),
            "evidence_label": claim["evidence_label"],
            "evidence_sentences": claim["evidence_sentences"]
        })

# Save to disk
with open("corpus.json", "w") as f:
    json.dump(corpus_clean, f)

with open("claims_val.json", "w") as f:
    json.dump(claims_clean, f)

print(f"Saved {len(corpus_clean)} documents")
print(f"Saved {len(claims_clean)} claims with ground truth")

Saved 5183 documents
Saved 338 claims with ground truth
